<h2>Implement advanced outlier detection techniques such as Isolation Forest, DBSCAN, and Local Outlier Factor (LOF), followed by robust validation and treatment strategies to enhance dataset quality and improve machine learning model performance.</h2>

In [1]:
import numpy as np
import pandas as pd

In [2]:
from sklearn.datasets import make_blobs

In [3]:
np.random.seed(42)

In [4]:
X_clean, _ = make_blobs(n_samples=300, centers=2, cluster_std=1.2, random_state=42)

In [5]:
outliers = np.random.uniform(low=-10, high=10, size=(20, 2))

In [6]:
df = pd.DataFrame(np.vstack([X_clean, outliers]), columns=['Feature_1', 'Feature_2'])

In [7]:
print("Dataset Shape:", df.shape)

Dataset Shape: (320, 2)


In [8]:
df.head()

,Feature_1,Feature_2
0,5.046075,1.474824
1,5.406190,-0.020654
2,-2.525394,7.745033
3,5.296396,1.730539
4,4.987609,4.463651


In [11]:
from sklearn.preprocessing import StandardScaler

In [12]:
from sklearn.ensemble import IsolationForest

In [13]:
from sklearn.cluster import DBSCAN

In [14]:
from sklearn.neighbors import LocalOutlierFactor

In [15]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[['Feature_1', 'Feature_2']])

In [16]:
iso_forest = IsolationForest(contamination=0.06, random_state=42)
df['iso_outlier'] = iso_forest.fit_predict(X_scaled)

In [18]:
dbscan = DBSCAN(eps=0.5, min_samples=5)
df['dbscan_outlier'] = np.where(dbscan.fit_predict(X_scaled) == -1, -1, 1)

In [19]:
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.06)
df['lof_outlier'] = lof.fit_predict(X_scaled)

In [20]:
print("Detection Summary (-1 indicates detected outlier):")
print(df[['iso_outlier', 'dbscan_outlier', 'lof_outlier']].value_counts())

Detection Summary (-1 indicates detected outlier):
iso_outlier  dbscan_outlier  lof_outlier
 1            1               1             299
-1           -1              -1              15
              1              -1               4
                              1               1
 1            1              -1               1
Name: count, dtype: int64


In [21]:
df['outlier_votes'] = (
    (df['iso_outlier'] == -1).astype(int) +
    (df['dbscan_outlier'] == -1).astype(int) +
    (df['lof_outlier'] == -1).astype(int)
)

In [22]:
df['confirmed_outlier'] = df['outlier_votes'] >= 2

In [23]:
print(f"Total Confirmed Outliers by Majority Vote: {df['confirmed_outlier'].sum()}")

Total Confirmed Outliers by Majority Vote: 19


In [24]:
df_cleaned = df[~df['confirmed_outlier']].copy()

In [25]:
df_capped = df.copy()

In [27]:
for col in ['Feature_1', 'Feature_2']:
    lower_limit = df[col].quantile(0.05)
    upper_limit = df[col].quantile(0.95)
    df_capped[col] = np.clip(df[col], lower_limit, upper_limit)

In [28]:
print(f"Original Shape: {df.shape}")

Original Shape: (320, 7)


In [29]:
print(f"Shape After Removal: {df_cleaned.shape}")

Shape After Removal: (301, 7)


In [32]:
print(f"Capped Dataset (Preserves Row Count): {df_capped.shape}")

Capped Dataset (Preserves Row Count): (320, 7)
